In [ ]:
import os
import torch
import glob as gb
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from scipy.signal import butter, lfilter

### ========== FILTERING ==========

In [ ]:
#Bandpass Filter Function
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data)

In [ ]:
#DE Extraction Function
def compute_de_features(eeg_data, fs=200):
    """
    eeg_data: shape [800, 62] — 200 timepoints * 4 seconds, 62 channels
    returns: Tensor [5, 62] — 5 bands: delta, theta, alpha, beta, gamma
    """
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 14),
        'beta': (14, 31),
        'gamma': (31, 50)
    }

    num_points, num_channels = eeg_data.shape
    de_features = np.zeros((len(bands), num_channels))

    for b_idx, (band_name, (low, high)) in enumerate(bands.items()):
        for ch in range(num_channels):
            signal = eeg_data[:, ch]
            filtered = bandpass_filter(signal, low, high, fs)
            variance = max(np.var(filtered), 1e-6)
            de = 0.5 * np.log(2 * np.pi * np.e * variance )  # numerical stability
            de_features[b_idx, ch] = de

    return de_features  # shape: [5, 62]

### ========== PRECOMPUTE SCRIPT ==========

In [ ]:
def precompute_and_save(csv_path, save_dir):
    """
    file_path: path to the .csv file that contains [49600 features + SessionLabel + SubjectLabel + EmotionLabel]
    """
    print(f"Loading: {csv_path.split('/')[-1]}")
    csv_name = csv_path.split('/')[-1].split('.')[0]

    df = pd.read_csv(csv_path)
    raw_data = df.values

    eeg_data = raw_data[:, :-3]
    session_labels = raw_data[:, -3]
    subject_labels = raw_data[:, -2]
    emotion_labels = raw_data[:, -1]

    num_trials = eeg_data.shape[0]
    output_features = []
    output_labels = []

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f'Processing {num_trials} samples...')

    for i in tqdm(range(num_trials)):
        raw_sample = eeg_data[i]
        reshaped = raw_sample.reshape(800, 62)  # [800, 62]
        de = compute_de_features(reshaped)  # [4, 62, 5]
        
        output_features.append(de)

        output_labels.append({
            'session': int(session_labels[i]),
            'subject': int(subject_labels[i]),
            'emotion': int(emotion_labels[i])
        })

    # Save features
    feature_tensor = torch.tensor(np.array(output_features), dtype=torch.float32)  #  [N, 5, 62]
    torch.save(feature_tensor, save_dir / f"{csv_name}_de_features.pt")
    
    # Save labels
    labels_df = pd.DataFrame(output_labels)
    labels_df.to_csv(save_dir / f"{csv_name}_labels.csv", index=False)
    
    
    print(f"Saved to: {save_dir / f'{csv_name}_de_features.pt'} and {save_dir / f'{csv_name}_labels.csv'}")


In [ ]:
# folder_path = "../SEEDiv_processed/filtered_csv"
# file_list = [file for file in gb.glob(os.path.join(folder_path,'*')) if file.endswith('.csv')]
# file =file_list[0]
# precompute_and_save(file, "../SEEDiv_processed/test")

### ========== RUN ==========

In [ ]:
folder_path = "../SEEDiv_processed/filtered_csv"
file_list = [file for file in gb.glob(os.path.join(folder_path,'*')) if file.endswith('.csv')]
print(file_list)
for file in file_list:
    precompute_and_save(file, "../SEEDiv_processed/precomputed")